In [ ]:
import pandas as pd 
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc
from pathlib import Path

In [ ]:
DATA = "../../../../data/"
SWED_RAW = "raw/swed_data/"
ER_VISITS = "copy_ehr_sahlgrenska_lab.parquet"

In [ ]:
pf = pq.ParquetFile(DATA +SWED_RAW+ ER_VISITS)
print(pf.metadata.num_rows, pf.num_row_groups)

In [ ]:
def parquet_chunks(path, batch_size=100_000, columns=None):
    """Yield the parquet file as pandas DataFrames of up to batch_size rows."""
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=batch_size, columns=columns):
        yield batch.to_pandas()

In [ ]:
df = pd.concat(parquet_chunks(DATA +SWED_RAW+ ER_VISITS), ignore_index=True)

In [ ]:
# pf_edvisit - visits to the er 
pf_edvisit = pq.ParquetFile(DATA +SWED_RAW+ ER_VISITS)

meta = pf_edvisit.metadata

# File-level metadata
print("=== file metadata ===")
print(f"created_by:      {meta.created_by}")
print(f"num_columns:      {meta.num_columns}")
print(f"num_rows:         {meta.num_rows}")
print(f"num_row_groups:   {meta.num_row_groups}")
print(f"format_version:   {meta.format_version}")
print(f"serialized_size:  {meta.serialized_size}")

# Schema, sorted alphabetically by column name
print("\n=== columns (sorted) ===")
schema = pf.schema_arrow
for field in sorted(schema, key=lambda f: f.name):
    print(f"{field.name:<40} {field.type}")